In [1]:
import os

In [3]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-'

In [26]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingPipelineConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    trained_data_path: Path
    params_epochs: int
    params_batch_size: int
    params_is_use_augmentation:bool 
    params_image_size:list
  

In [27]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml, create_directories

In [28]:
import tensorflow as tf

In [ ]:
class configuartionManager:
    def __init__(self, config_file_path = config_file_path, params_file_path = params_file_path):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_training_pipeline_config(self) -> TrainingPipelineConfig:
        training_config = self.config.training
        prepare_base_model= self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"KidneyData")
        create_directories([Path(training_config.root_dir), Path(training_data)])
        training = training_config

        
        training_pipeline_config = TrainingPipelineConfig(
            root_dir=Path(training_config.root_dir),
            trained_model_path=Path(training_config.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            trained_data_path=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_use_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )
        return training_pipeline_config

In [30]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [31]:
class Training:
    def __init__(self, config: TrainingPipelineConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)  
        dataflow_kwargs = dict(target_size=self.config.params_image_size[:-1], batch_size=self.config.params_batch_size, interpolation="bilinear")
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(directory=self.config.trained_data_path, subset="validation", shuffle=False, **dataflow_kwargs)
        if self.config.params_is_use_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(directory=self.config.trained_data_path, subset="training", shuffle=True, **dataflow_kwargs)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )
        self.save_model(path=self.config.trained_model_path, model=self.model)

In [32]:
import traceback

try:
    training_pipeline_config = configuartionManager().get_training_pipeline_config()
    training = Training(config=training_pipeline_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    print(f"Error: {e}")
    traceback.print_exc()

2026-08-04 14:23:34,850 - cnnclassifier - INFO - yaml file: config\config.yaml loaded successfully
2026-08-04 14:23:34,853 - cnnclassifier - INFO - yaml file: params.yaml loaded successfully
2026-08-04 14:23:34,854 - cnnclassifier - INFO - created directory at: artifacts
2026-08-04 14:23:34,856 - cnnclassifier - INFO - created directory at: artifacts\training
2026-08-04 14:23:34,857 - cnnclassifier - INFO - created directory at: artifacts\data_ingestion\unzip\Training
2026-08-04 14:23:35,413 - tensorflow - WARNING - TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
2026-08-04 14:23:35,417 - absl - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Error: The PyData

Traceback (most recent call last):
  File "C:\Users\abhin\AppData\Local\Temp\ipykernel_16064\2617095815.py", line 8, in <module>
    training.train()
    ~~~~~~~~~~~~~~^^
  File "C:\Users\abhin\AppData\Local\Temp\ipykernel_16064\486101777.py", line 36, in train
    self.model.fit(
    ~~~~~~~~~~~~~~^
        self.train_generator,
        ^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        validation_data=self.valid_generator
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\abhin\Desktop\Mlops\Deep-Learning-Kidney-Tumor-Classification-\kidneyclassification\Lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "c:\Users\abhin\Desktop\Mlops\Deep-Learning-Kidney-Tumor-Classification-\kidneyclassification\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py", line 311, in get_tf_dataset
    raise ValueError("The PyDataset has length 0")
ValueError: The PyDataset ha